In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from matplotlib.lines import Line2D
from matplotlib_venn import venn3  # pip install matplotlib-venn if needed
import importlib
import numpy as np
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import seaborn as sns

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.plotting as plott
import src.data_loading as exio
import src.analysis_tools as at
import matplotlib as mpl


plt.style.use('default')
# %matplotlib qt
%matplotlib inline

In [ ]:
# most ogf fish respond pretty well to stimuli in the back.... normally rigth side!

# comments about fish quality:
#L433_f02: good fish, but the response is mostly one side
# Exp1_good_fish= L433_f02,L433_f04,f453_f09,f453_f11
# Exp2_good_fih = fish_ids = [ "L472_f02", "L472_f03",  "L472_f04", "L472_f06"}


In [ ]:
# ==== CONFIG ================================================================
importlib.reload(exio)
fish_id = "L453_f08"
experiment_name = 'Exp_1_flickering' #"Exp_2_rocking_1" #'Exp_1_flickering' #
main_path = Path(r"D:\Alejandro\Data\OneDrive - Université de Lausanne\Lab\Data\2p")
stimuli_main_path = Path(r"D:\Alejandro\Data\OneDrive - Université de Lausanne\Lab\Data\stimuli")

# stimuli_main_path = Path(r"\\nasdcsr.unil.ch\RECHERCHE\FAC\FBM\CIG\jlarsch\default\D2c\Alejandro\2p")
fps_2p = 2.0
selected_blocks = [f"B{n}" for n in range(1, 3)]
t_pre_s  = 5.0   # time before onset to include
t_post_s = 32.0  # time after onset to include
# ==== LOAD EVERYTHING IN ONE GO ============================================
results = exio.load_2p_experiment(
    fish_id=fish_id,
    experiment_name=experiment_name,
    main_path=main_path,
    stimuli_main_path=stimuli_main_path,
    fps_2p=fps_2p,
    selected_blocks=selected_blocks,
)

dfof              = results["dfof"]
stimuli_durations = results["stimuli_durations"]
exp_log           = results["adjusted_log"]
stimuli_trace_60  = results["stimuli_trace_60"]
stimuli_table     = results["stimuli_table"]
stimuli_id_map    = results["stimuli_id_map"]
duration_2p_block_sec = results["duration_2p_block_sec"]
frames_per_block       = results["frames_per_block"]
paths                  = results["paths"]
raster                 = results["raster"]
dfof_center =           results["deltaF_center"]
dFoF_merged_map =       results["dFoF_merged_map"]
plane_ids =             results["plane_ids"]
mean_imgs =             results["mean_imgs"]
stat_per_plane =        results["stat_per_plane"]


In [ ]:
# detect Z-score, as a base line a contatenated trace of traces pre-stimulu

# detect onset of stimuli
res = at.build_trial_aligned_traces(
        dfof=dfof,
        stimuli_trace_60=stimuli_trace_60,
        fps_2p=fps_2p,
        t_pre_s=t_pre_s,
        t_post_s=t_post_s,
        stimuli_id_map=stimuli_id_map,  # or None
        verbose=True,
    )
onsets_by_id= res["onsets_by_id"]

zres = at.zscore_dfof_from_prestim_baseline(
    dfof=dfof,
    onsets_by_id=onsets_by_id,
    fps_2p=fps_2p,
    baseline_s=10.0,
    assume_time_by_neuron=True,
    verbose=True,
)

z_traces = zres["z_traces"]

prefix = paths["prefix"]
folder_zcore = paths["plots_path"] / 'z_core'
folder_zcore.mkdir(parents=True, exist_ok=True)
out_zcore = folder_zcore / f"{prefix}_zcore.npz"

np.savez(
    out_zcore,
    z_traces=zres["z_traces"],
    baseline_mean=zres["baseline_mean"],
    baseline_std=zres["baseline_std"],
    valid_onsets=zres["valid_onsets"]
)

print(f"Saved z_traces to {out_zcore}")


In [ ]:
# select colors and linestyles for stimuli
# --- Set A (default) ----------------------------------------------------------
colors_A = {
    'LLB': sns.color_palette('Blues', 8)[7],
    'RLB': sns.color_palette('Greens', as_cmap=True)(0.6),
    'FLB': sns.color_palette('Blues', 8)[7],
    'FRB': sns.color_palette('Greens', as_cmap=True)(0.6),
    'FL1': sns.color_palette('PuRd', as_cmap=True)(0.9),
    'FL2': sns.color_palette('PuRd', as_cmap=True)(0.6),
    'FL3': sns.color_palette('dark:#404040', 10)[7],
    'FR1': sns.color_palette('PuRd', as_cmap=True)(0.9),
    'FR2': sns.color_palette('PuRd', as_cmap=True)(0.6),
    'FR3': sns.color_palette('dark:#404040', 10)[7],
}
linestyles_A = {
    'FLB': '--',
    'FRB': '--',
    'FL1': '--',
    'FL2': '--',
    'FL3': '--',
}

# --- Set B (rocking experiment) ----------------------------------------------
colors_B = {
    'LB':   sns.color_palette('Blues', 8)[7],
    'RR1':  sns.color_palette('Greens', as_cmap=True)(0.6),
    'RB':   sns.color_palette('Blues', 8)[7],
    'RL1':  sns.color_palette('Greens', as_cmap=True)(0.6),
    'RR2':  sns.color_palette('PuRd', as_cmap=True)(0.9),
    'RR3':  sns.color_palette('PuRd', as_cmap=True)(0.6),
    'RLR1': sns.color_palette('dark:#404040', 10)[7],
    'RL2':  sns.color_palette('PuRd', as_cmap=True)(0.9),   # fixed case from 'Rl2' → 'RL2'
    'RL3':  sns.color_palette('PuRd', as_cmap=True)(0.6),
    'RLR2': sns.color_palette('dark:#404040', 10)[7],
}
linestyles_B = {
    'LB': '--',
    'RL1': '--',
    'RL2': '--',
    'RL3': '--',
    'RLR2': '--',
}

# --- Select set based on experiment name --------------------------------------
if experiment_name == 'Exp_2_rocking_1':
    stimuli_colors = colors_B
    stimuli_linestyles = linestyles_B
else:
    stimuli_colors = colors_A
    stimuli_linestyles = linestyles_A

print("Using colors for:", experiment_name)
print(stimuli_linestyles)

# --- choose stimulus-aligned windows ---
window_pre = 5  # seconds before stimulus
window_post = 17  # seconds after stimulus


# define stimuli order based on experiment
stimuli_ordered_A = ['LLB', 'FLB', 'RLB' , 'FRB','FL1', 'FL2', 'FL3', 'FR1', 'FR2', 'FR3']  # Order of
stimuli_ordered_B = ['LB','RB','RR1','RR2','RR3','RL1','RL2','RL3','RLR1','RLR2' ]

if experiment_name == 'Exp_2_rocking_1':
    stimuli_ordered = stimuli_ordered_B
else:
    stimuli_ordered = stimuli_ordered_A


In [ ]:
fig, ax, order = plott.plot_sorted_chunks_single_mode(
    dfof=z_traces,
    exp_log=exp_log,
    stimuli_durations=stimuli_durations,
    stimuli_colors=stimuli_colors,
    stimuli_linestyles=stimuli_linestyles,
    stimuli_ordered=stimuli_ordered,
    fps_2p=fps_2p,
    window_pre=window_pre,
    window_post=window_post,
    sort_mode="corravg",        # or 'pca', 'hier', 'corravg', max_intensity, kmeans, none(unsorted) ...
    n_clusters=5,
    random_state=42,
    average_across_repeats=False, # 👈 False to keep all repeats True to average across repeats
    fish_id=fish_id,
    neuron_order=None,       # 👈 custom order (optional)
    sort_label=None,         # 👈 label to show in the title
)
plt.show()


# Question: the cells that respond to flickering stimuli are the same that respond to bouts stimuli?

# ==== IDENTIFY FLICKERING-RESPONSIVE CELLS ==================================
# Get the stimuli IDs and onsets where stimuli began for flickering stimuli
# Get significant traces aligned to flickering stimuli



In [ ]:
## Generate trial-aligned traces regarding stimuli
# Build trial-aligned traces, i.e., snippets of dF/F around each stimulus onset
# you define the pre- and post- time windows (in seconds) onset_begining of stimuli___moving.
t_pre_s  = 5.0   # time before onset to include
t_post_s = 29.0  # time after onset to include

res = at.build_trial_aligned_traces(
    dfof=dfof,
    stimuli_trace_60=stimuli_trace_60,
    fps_2p=fps_2p,
    t_pre_s=t_pre_s,
    t_post_s=t_post_s,
    stimuli_id_map=stimuli_id_map,  # or None
    verbose=True
)

trial_aligned_traces = res["trial_aligned_traces"] # dict: stim_id -> (n_neurons, n_trials_frames_per_trial) Df/F
stimuli_ids          = res["stimuli_ids"]         # list of stim_ids used
stimuli_names        = res["stimuli_names"]       # list of stim_names used
win_length           = res["win_length"]          # n_frames_per_trial
stimuli_trace        = res["stimuli_trace"]       # (n_frames_total,) stimulus trace used for alignment
onsets_by_id         = res["onsets_by_id"]        # dict: stim_id -> (n_trials,) onset frames (global indices)


In [ ]:
stim_order = [9,10,1,2,3,4,5,6,7]
combine_mode = "concat"  # mean or concat
# combine_mode = "concat"   # or "mean"
# # M_all = build_matrix_all_fish(all_fish_data, stim_order,
#                               combine_mode=combine_mode)
fig, ax, im, neuron_order = plott.plot_allfish_flat_raster(
    data=z_traces.T,
    trial_aligned_traces=trial_aligned_traces,
    stim_order=stim_order,
    stimuli_id_map=stimuli_id_map,
    stimuli_durations=stimuli_durations,
    stimuli_colors=stimuli_colors,
    stimuli_linestyles=stimuli_linestyles,
    fps_2p=fps_2p,
    t_pre_s=t_pre_s,
    combine_mode=combine_mode,
    sort_mode='corravg',      # or "pca", "hier", etc,unsorted
    n_clusters=3,
    random_state=0,
    neuron_order=None,       # or pass your own `neuron_order`
    sort_label=None,         # optional label override
    is_binary=False,         # True if you pass a 0/1 significance matrix
    show_mean_trace=True,
    figsize=(12, 8),
    fish_id="all_fish",
)
plt.show()


In [ ]:
unique_vals = np.unique(stimuli_trace)
print(onsets_by_id )

In [ ]:
'''First by big groups (response pattern to the 3 stimuli): In this order: Evoked in all 3 → ev_123
Evoked in 1&2 only → ev_12_only Evoked in 1&3 only → ev_13_only Evoked in 2&3 only → ev_23_only
Evoked only in 1 → ev_1_only Evoked only in 2 → ev_2_only Evoked only in 3 → ev_3_only then the same 7 patterns for tardive (td_123, td_12_only, …, td_3_only), and finally: no_response → neurons that are not evoked or tardive to any of the 3 stimuli.
Then, inside each group:  For all evoked/tardive groups: neurons are sorted by their mean onset across the 3 stimuli (earliest → latest; neurons with no onset = NaN go at the end of that group).

For the no_response group: neurons are sorted by onset to the extra stimulus stim_id_for_noresp_sort
(again: earliest → latest; NaN onsets go at the end of the no_response group).

'''''

def build_neuron_order_groupwise_onset(
    response_type_by_id,
    onset_sec_by_stim,
    stim_ids_for_pattern,
    stim_id_for_noresp_sort=None,
):
    """
    Group neurons by response pattern to three stimuli, and sort within each
    group by onset:

    - For evoked/tardive groups (defined by `stim_ids_for_pattern`), sort by
      the neuron's *mean onset* across those stimuli where it responds.
      (Implemented via np.nanmean across the 3 onset arrays.)
    - For the no_response group (no evoked/tardive to those 3), sort by onset
      to another stimulus `stim_id_for_noresp_sort` (e.g. 10).
    - Within each group: finite onset first (ascending), then NaNs.

    Parameters
    ----------
    response_type_by_id : dict[int, np.ndarray]
        stim_id -> (n_neurons,) with values:
          0 = no response, 1 = evoked, 2 = tardive.
    onset_sec_by_stim : dict[int, np.ndarray]
        stim_id -> (n_neurons,) onset latencies (seconds).
        Use np.nan where no onset / no response.
        Must contain all `stim_ids_for_pattern` and (if used)
        `stim_id_for_noresp_sort`.
    stim_ids_for_pattern : list[int]
        3 stimulus IDs used to define response patterns, e.g. [5, 6, 7].
    stim_id_for_noresp_sort : int or None
        Stimulus ID whose onsets are used to sort the no_response group,
        e.g. 10. If None, no_response will also use the mean onset across
        `stim_ids_for_pattern` (usually they’ll all be NaN).

    Returns
    -------
    neuron_order : np.ndarray
        Final neuron order (indices 0..n_neurons-1 in the chosen order).
    groups : dict[str, np.ndarray]
        Mapping group_name -> sorted indices (in that group).
    mean_onset_pattern : np.ndarray
        (n_neurons,) mean onset across `stim_ids_for_pattern` (nanmean).
    """

    # --- stack response types for pattern stimuli ---
    resp_matrix = np.stack(
        [response_type_by_id[sid] for sid in stim_ids_for_pattern],
        axis=1
    )  # shape: (n_neurons, 3)

    n_neurons = resp_matrix.shape[0]

    # Boolean matrices: evoked and tardive for each of the 3 stimuli
    ev = (resp_matrix == 1)
    td = (resp_matrix == 2)

    # --- compute mean onset across the 3 pattern stimuli, per neuron ---
    # shape: (n_neurons, 3)
    onset_stack_pattern = np.stack(
        [onset_sec_by_stim[sid] for sid in stim_ids_for_pattern],
        axis=1
    )

    # nanmean: ignores NaNs. If all 3 are NaN, result is NaN.
    mean_onset_pattern = np.nanmean(onset_stack_pattern, axis=1)

    # onset used for sorting no_response group
    onset_noresp = None
    if stim_id_for_noresp_sort is not None:
        onset_noresp = onset_sec_by_stim[stim_id_for_noresp_sort]

    used = np.zeros(n_neurons, dtype=bool)
    groups = {}
    order_list = []

    def add_group(name, mask, use_noresp_onset=False):
        nonlocal used, groups, order_list

        idx = np.where(mask & ~used)[0]
        if idx.size == 0:
            return

        # choose which onset to use for this group
        if use_noresp_onset and (onset_noresp is not None):
            base_onset = onset_noresp
        else:
            base_onset = mean_onset_pattern

        group_onset = base_onset[idx]

        # finite onset first, then NaNs
        finite_mask = np.isfinite(group_onset)
        finite_local = np.where(finite_mask)[0]
        nan_local   = np.where(~finite_mask)[0]

        if finite_local.size > 0:
            order_finite = np.argsort(group_onset[finite_mask])
            sorted_local = np.concatenate([
                finite_local[order_finite],
                nan_local,
            ])
        else:
            # all NaN: keep original relative order (or just nan_local)
            sorted_local = nan_local

        idx_sorted = idx[sorted_local]

        groups[name] = idx_sorted
        order_list.append(idx_sorted)
        used[idx_sorted] = True

    # indices in second axis
    s0, s1, s2 = 0, 1, 2

    # ========= EVOKED (type 1) GROUPS =========
    add_group("ev_123",      ev[:, s0] & ev[:, s1] & ev[:, s2])
    add_group("ev_12_only",  ev[:, s0] & ev[:, s1] & ~ev[:, s2])
    add_group("ev_13_only",  ev[:, s0] & ~ev[:, s1] & ev[:, s2])
    add_group("ev_23_only",  ~ev[:, s0] & ev[:, s1] & ev[:, s2])
    add_group("ev_1_only",   ev[:, s0] & ~ev[:, s1] & ~ev[:, s2])
    add_group("ev_2_only",   ~ev[:, s0] & ev[:, s1] & ~ev[:, s2])
    add_group("ev_3_only",   ~ev[:, s0] & ~ev[:, s1] & ev[:, s2])

    # ========= TARDIVE (type 2) GROUPS =========
    add_group("td_123",      td[:, s0] & td[:, s1] & td[:, s2])
    add_group("td_12_only",  td[:, s0] & td[:, s1] & ~td[:, s2])
    add_group("td_13_only",  td[:, s0] & ~td[:, s1] & td[:, s2])
    add_group("td_23_only",  ~td[:, s0] & td[:, s1] & td[:, s2])
    add_group("td_1_only",   td[:, s0] & ~td[:, s1] & ~td[:, s2])
    add_group("td_2_only",   ~td[:, s0] & td[:, s1] & ~td[:, s2])
    add_group("td_3_only",   ~td[:, s0] & ~td[:, s1] & td[:, s2])

    # ========= NON-RESPONSIVE TO THESE 3 STIMULI =========
    no_resp_mask = ~(ev.any(axis=1) | td.any(axis=1))
    # For this group, use onset to the extra stimulus (e.g. stim 10)
    add_group("no_response", no_resp_mask, use_noresp_onset=True)

    # Final order
    if len(order_list) > 0:
        neuron_order = np.concatenate(order_list)
    else:
        neuron_order = np.arange(n_neurons)

    return neuron_order, groups, mean_onset_pattern


In [ ]:
# Classify responses based on raster (significant_traces) and stimulus onsets, IT evaluates for each neuron
# whether it has evoked or tardive responses to each stimulus
# based on criteria like minimum run length and fraction of trials.

response_type_by_id, onset_sec_by_id = at.classify_responses_from_raster(
    raster=raster,
    onsets_by_id=onsets_by_id,
    stimuli_id_map=stimuli_id_map,
    stimuli_durations=stimuli_durations,
    fps_2p=2.0,
    response_window_sec=20.0,  # 0–20 s after each onset
    min_run_sec=2.0,           # ≥ 3 s consecutive 1s
    min_fraction_trials=0.5,   # ≥ 50% of trials
)

#{'FL1': 1, 'FL2': 2, 'FL3': 3, 'FLB': 4, 'FR1': 5, 'FR2': 6, 'FR3': 7, 'FRB': 8, 'LLB': 9, 'RLB': 10}

# 3 stimuli that define the pattern (e.g. FR1, FR2, FR3)
stim_ids_for_pattern = [5, 6, 7]

# Build a dict of onsets per stimulus (you may already have this):
# onset_sec_by_stim = {1: onset_FL1, 2: onset_FL2, ..., 10: onset_RLB, ...}

# Stimulus you want to use for sorting the no_response group (e.g. RLB = 10)
stim_id_for_noresp_sort = 10

neuron_order, groups, mean_onset_pattern = build_neuron_order_groupwise_onset(
    response_type_by_id=response_type_by_id,
    onset_sec_by_stim=onset_sec_by_id,
    stim_ids_for_pattern=stim_ids_for_pattern,
    stim_id_for_noresp_sort=stim_id_for_noresp_sort,
)

print("Group sizes:")
for name, idx in groups.items():
    print(f"{name:12s}: {idx.size} neurons")

# Apply this order to your dF/F
dfof_sorted = dfof[neuron_order, :]


In [ ]:
importlib.reload(plott)

fig, ax, order = plott.plot_sorted_chunks_single_mode(
    dfof=raster,
    exp_log=exp_log,
    stimuli_durations=stimuli_durations,
    stimuli_colors=stimuli_colors,
    stimuli_linestyles=stimuli_linestyles,
    stimuli_ordered=stimuli_ordered,
    fps_2p=fps_2p,
    window_pre=window_pre,
    window_post=window_post,
    sort_mode="corravg",        # or 'pca', 'hier', 'corravg', max_intensity, kmeans, none(unsorted) ...
    n_clusters=3,
    random_state=42,
    average_across_repeats=True, # 👈 False to keep all repeats True to average across repeats
    fish_id=fish_id,
    neuron_order=neuron_order,       # 👈 custom order (optional)
    sort_label=None,# 👈 label to show in the title
    is_binary = True,
)
plt.show()

In [ ]:
# Venn diagram of responsiveness to 3 stimuli.

def plot_venn_3stim(response_type_by_id, stim_ids=(5, 6, 7)):
    # 1) Boolean masks: neuron is responsive (1 or 2) to each stim
    resp = {sid: (response_type_by_id[sid] != 0) for sid in stim_ids}

    # 2) Convert to sets of neuron indices
    sets = [set(np.where(resp[sid])[0]) for sid in stim_ids]
    A, B, C = sets
    s1, s2, s3 = stim_ids

    # 3) Print counts per region
    only_A = A - B - C
    only_B = B - A - C
    only_C = C - A - B
    AB_only = (A & B) - C
    AC_only = (A & C) - B
    BC_only = (B & C) - A
    ABC     = A & B & C

    print(f"Only {s1}: {len(only_A)}")
    print(f"Only {s2}: {len(only_B)}")
    print(f"Only {s3}: {len(only_C)}")
    print(f"{s1} & {s2} only: {len(AB_only)}")
    print(f"{s1} & {s3} only: {len(AC_only)}")
    print(f"{s2} & {s3} only: {len(BC_only)}")
    print(f"{s1} & {s2} & {s3}: {len(ABC)}")

    # 4) Plot Venn diagram
    plt.figure(figsize=(5, 5))
    venn3(sets, set_labels=[f"stim {s1}", f"stim {s2}", f"stim {s3}"])
    plt.title("Responsive neurons to stimuli 5, 6, 7")
    plt.show()

# Use it like this:
plot_venn_3stim(response_type_by_id, stim_ids=(5, 6, 7))


In [ ]:
#{'FL1': 1, 'FL2': 2, 'FL3': 3, 'FLB': 4, 'FR1': 5, 'FR2': 6, 'FR3': 7, 'FRB': 8, 'LLB': 9, 'RLB': 10}

def compute_left_right_index(mean_traces,
                             left_stim,
                             right_stim,
                             frame_window,
                             kept_cells=None,
                             eps=1e-9):
    """
    mean_traces[stim]: (n_cells, n_frames)
    frame_window: (start_frame, end_frame) for averaging
    kept_cells: optional index/mask of cells to use
    """
    start, end = frame_window

    M_left  = mean_traces[left_stim][:, start:end]   # (n_cells, win_len)
    M_right = mean_traces[right_stim][:, start:end]

    # Optional subset of cells
    if kept_cells is not None:
        M_left  = M_left[kept_cells]
        M_right = M_right[kept_cells]

    # Scalar response per neuron (mean over time window)
    L = np.nanmean(M_left,  axis=1)   # (n_cells_sel,)
    R = np.nanmean(M_right, axis=1)

    # Tuning index in [-1, 1]
    TI = (R - L) / (R + L + eps)

    return L, R, TI


In [ ]:
kept_neuron_indices = results["kept_neuron_indices"]
# average the  activity of neurons in repetition per stimuli

# Containers (per stimulus → arrays of shape (n_kept,))
mean_traces = {}   # trial-averaged timecourses (kept neurons)
pre_frames  = int(round(t_pre_s * fps_2p))
post_frames = int(round(t_post_s * fps_2p))
win_lenght = pre_frames + post_frames

for stim in stimuli_ids:
    X = trial_aligned_traces[stim]                      # (n_neurons, win_lenght, n_trials)
    Xkept = X[kept_neuron_indices, :, :]                   # filter kept neurons
    mean_trace = np.nanmean(Xkept, axis=2)                # (kept_neurons, win_lenght)
    mean_traces[stim] = mean_trace


#{'FL1': 1, 'FL2': 2, 'FL3': 3, 'FLB': 4, 'FR1': 5, 'FR2': 6, 'FR3': 7, 'FRB': 8, 'LLB': 9, 'RLB': 10}

frame_window = (pre_frames, win_lenght)  # for example

L, R, TI = compute_left_right_index(
    mean_traces=mean_traces,
    left_stim=1,   # e.g., FLB = 4
    right_stim=2,
    frame_window=frame_window,
    kept_cells=None,   # or None
)


In [ ]:
def classify_lr_tuning(TI, thr=0.2):
    """
    thr: how strong the index must be to call it tuned
    """
    right_pref = TI >  thr
    left_pref  = TI < -thr
    non_sel    = np.abs(TI) <= thr
    return left_pref, right_pref, non_sel

left_pref, right_pref, non_sel = classify_lr_tuning(TI, thr=0.2)

idx_left     = np.where(left_pref)[0]
idx_right    = np.where(right_pref)[0]
idx_non_sel  = np.where(non_sel)[0]

n_left  = left_pref.sum()
n_right = right_pref.sum()
n_ns    = non_sel.sum()

print("Left-preferring:", n_left)
print("Right-preferring:", n_right)
print("Non-selective:", n_ns)


In [ ]:
importlib.reload(plott)
neuron_order=  np.argsort(TI)[::-1]

fig, ax, order = plott.plot_sorted_chunks_single_mode(
    dfof=raster,
    exp_log=exp_log,
    stimuli_durations=stimuli_durations,
    stimuli_colors=stimuli_colors,
    stimuli_linestyles=stimuli_linestyles,
    stimuli_ordered=stimuli_ordered,
    fps_2p=fps_2p,
    window_pre=window_pre,
    window_post=window_post,
    sort_mode="hier",        # or 'pca', 'hier', 'corravg', max_intensity, kmeans, none(unsorted) ...
    n_clusters=3,
    random_state=42,
    average_across_repeats=True, # 👈 False to keep all repeats True to average across repeats
    fish_id=fish_id,
    neuron_order=neuron_order,       # 👈 custom order (optional)
    sort_label=None,# 👈 label to show in the title
    is_binary = True,
)
plt.show()

In [ ]:
##map neurons in thalamus


### I will plot for onset,

# map of index from dFoF_merged
map_df = dFoF_merged_map
print(map_df.head())


# I am taking the same mask to filter kept indices
kept_indices=results["kept_neuron_indices"]

# e.g.: BPI = np.load(paths['dfof_dir'] / "L433_f02_BPI_kept.npy")
onset = TI  # <--- put your actual variable here
print("onset shape:", onset.shape)
assert onset.shape[0] == kept_indices.shape[0], "onset and kept_indices must have same length"
# DataFrame for kept neurons: global_col ↔ onset or other continuous value
sel_df = pd.DataFrame({
    "global_col": kept_indices,
    "onset": onset
})

# Join with map to add plane and filtered_roi_index
merged = sel_df.merge(
    map_df[["plane", "filtered_roi_index", "global_col"]],
    on="global_col",
    how="left"
)
print(sel_df.head())
print("Merged kept+map example:")
print(merged.head())
assert merged["filtered_roi_index"].isna().sum() == 0, "Some kept neurons not in map_df!"

# Initialize dict: plane -> list of (k, BPI)
entries_per_plane = {pid: [] for pid in plane_ids}

for _, row in merged.iterrows():
    plane_label = row["plane"]                 # 'plane0', 'plane1', ...
    k = int(row["filtered_roi_index"])         # filtered ROI index
    value = float(row["onset"])              # value
# store (Suite2P ROI index, BPI) in the list for that plane
    entries_per_plane[plane_label].append((k, value))

for pid in plane_ids:
    print(pid, "n entries:", len(entries_per_plane[pid]))

In [ ]:
from matplotlib.colors import Normalize
import matplotlib.cm as cm

lr_norm = Normalize(vmin=-0.6, vmax=0.6)

plot_bpi_rois_by_plane(
    plane_ids=plane_ids,
    mean_imgs=mean_imgs,
    stat_per_plane=stat_per_plane,
    entries_per_plane=entries_per_plane,
    value_label="Left Right Index",
    value_norm=lr_norm,
    cmap=cm.coolwarm,
)

In [ ]:
def compare_stim8_with_567(response_type_by_id):
    """
    Compare responsiveness to stim 8 vs stims 5, 6, 7.
     #{'FL1': 1, 'FL2': 2, 'FL3': 3, 'FLB': 4, 'FR1': 5, 'FR2': 6, 'FR3': 7, 'FRB': 8, 'LLB': 9, 'RLB': 10}
    Returns
    -------
    idx_8_and_567 : np.ndarray
        Neuron indices that are responsive to stim 8
        AND to at least one of stim 5, 6, or 7.
    idx_any_567 : np.ndarray
        Neuron indices that are responsive to at least one
        of stim 5, 6, or 7 (1 or 2).
    masks : dict
        Dictionary with boolean masks if you want them.
    """
    resp_by_id = {}  # will hold boolean masks

    for stim_id, arr in response_type_by_id.items():
    # arr is (n_neurons,) for this stimulus
        resp_by_id[stim_id] = (arr != 0)

    # responsive to at least one of 5, 6, 7
    resp_567 = resp_by_id[5] | resp_by_id[6] | resp_by_id[7]# (n_neurons,)
    resp_8_9 = resp_by_id[8] | resp_by_id[9]
    resp_123 = resp_by_id[1] | resp_by_id[2] | resp_by_id[3]
    resp_4_10 = resp_by_id[4] | resp_by_id[10]
    # neurons that respond to 8 AND to at least one of 5/6/7
    mask_8_and_567 = resp_8_9 & resp_567
    mask_4_and_123 = resp_4_10 & resp_123

    idx_8_and_567 = np.where(mask_8_and_567)[0]
    idx_any_567   = np.where(resp_567)[0]
    idx_any_123   = np.where(resp_123)[0]
    idx_4_and_123 = np.where(mask_4_and_123)[0]

    ## for rocking_experiment
    #{'LB': 1, 'RB': 2, 'RL1': 3, 'RL2': 4, 'RL3': 5, 'RLR1': 6, 'RLR2': 7, 'RR1': 8, 'RR2': 9, 'RR3': 10}
    resp_345 = resp_by_id[4] | resp_by_id[5] | resp_by_id[3]
    resp_8_9_10 = resp_by_id[8] | resp_by_id[9] | resp_by_id[10]
    resp_67 = resp_by_id[6] | resp_by_id[7]
    resp_12 = resp_by_id[1] | resp_by_id[2]

    indx_12=  np.where(resp_12 )[0]
    indx_67= np.where(resp_67)[0]
    indx_345  = np.where(resp_345 )[0]
    indx_8_9_10 = np.where(resp_8_9_10)[0]




    masks = {
        "resp_8": resp_by_id[8],
        "resp_567": resp_567,
        "8_and_567": mask_8_and_567,
    }

    return idx_8_and_567, idx_any_567, idx_any_123, idx_4_and_123, masks, indx_345, indx_8_9_10, indx_12,  indx_67

idx_8_and_567, idx_any_567, idx_any_123, idx_4_and_123, masks, indx_345, indx_8_9_10, indx_12,  indx_67= compare_stim8_with_567(response_type_by_id)

print("Neurons responsive to 8 AND (5 or 6 or 7):", idx_8_and_567.shape[0])
print("Neurons responsive to at least one of 5,6,7:", idx_any_567.shape[0])

In [ ]:
kept_neuron_indices = results["kept_neuron_indices"]
# average the  activity of neurons in repetition per stimuli
dt = 1.0 / float(fps_2p) #time per frame (the sampling interval) in seconds
# Containers (per stimulus → arrays of shape (n_kept,))
mean_traces = {}   # trial-averaged timecourses (kept neurons)
peaks = {}
aucs  = {}
averages = {}

for stim in stimuli_ids:
    X = trial_aligned_traces[stim]                      # (n_neurons, win_lenght, n_trials)
    Xkept = X[kept_neuron_indices, :, :]                   # filter kept neurons
    mean_trace = np.nanmean(Xkept, axis=2)                # (kept_neurons, win_lenght)
    mean_traces[stim] = mean_trace

    # Stimulus window (without pre stim frames)
    Y = mean_trace[:, pre_frames:win_lenght]                # (n_kept, n_post)
    peaks[stim] = np.nanmax(Y, axis=1) # peak ΔF/F
    aucs[stim]  = np.trapezoid(np.nan_to_num(Y), dx=dt, axis=1)  # AUC (ΔF/F * s), NaNs→0
    averages[stim] = np.nanmean(Y, axis=1)

In [ ]:
stimuli_whanted = [5, 6, 7,8]  # first, second, third "class"

for stim in stimuli_whanted:
    M = mean_traces[stim]
    st = stimuli_ids [stim]
    name = stimuli_names[stim]
print(M,st,name)

In [ ]:
### for plotting just some of the stimuli...
  #{'FL1': 1, 'FL2': 2, 'FL3': 3, 'FLB': 4, 'FR1': 5, 'FR2': 6, 'FR3': 7, 'FRB': 8, 'LLB': 9, 'RLB': 10}
stimuli_wanted = [4, 5, 6, 7]  # positions in the list (0-based!)

M = {}                 # new dict with selected mean_traces
new_stimuli_ids = []   # new list of ids
new_stimuli_names = [] # new list of names

for pos in stimuli_wanted:
    # id in your list at that position
    stim_id = stimuli_ids[pos]
    stim_name = stimuli_names[pos]

    # take the correct array from the dict
    M[stim_id] = mean_traces[stim_id]

    # store id and name in new lists
    new_stimuli_ids.append(stim_id)
    new_stimuli_names.append(stim_name)

print("new_stimuli_ids:  ", new_stimuli_ids)
print("new_stimuli_names:", new_stimuli_names)
# example: check one of the matrices
print("shape of first M:", M[new_stimuli_ids[0]].shape)


In [ ]:

fig, ax, used_colors, out_path = plott.plot_stimulus_means(
    mean_traces=M,
    stimuli_ids=new_stimuli_ids,
    stimuli_names=new_stimuli_names,
    title_prefix="",
    fps_2p=fps_2p,
    t_post_s=t_post_s,
    t_pre_s=t_pre_s,
    stimuli_durations= stimuli_durations,
    plots_path=paths["plots_path"],       # Path object or str
    prefix=paths['prefix'],               # e.g., "exp12_mouseA"
    dpi=600,
    save=False,
    stimuli_colors=stimuli_colors,            # <— pass styles here
    stimuli_linestyles=stimuli_linestyles,
    close_after=False,
    kept_cells = idx_8_and_567,
    comment="jhghjg" # for saving
)


In [ ]:
# plane_ids = sorted(map_df['plane'].unique())
# print("Plane IDs:", plane_ids)
#
#
# fish = f"{fish_id}_{experiment_name}"
# # --- paths you already have ---
# # main_path, fish, paths, plots_path must already exist in your notebook
# planes_dir = main_path / fish / "03_analysis" / "functional" / "suite2P"
#
# # We'll get plane IDs from the map CSV later, to be safe.
#
# mean_imgs = {}
# stat_per_plane = {}
#
#
# for pid in plane_ids:  # e.g. 'plane0', 'plane1', ...
#     plane_path = planes_dir / pid
#
#     # Load ops and stat (Suite2P format)
#     ops = np.load(plane_path / "ops.npy", allow_pickle=True).item()
#     stat = np.load(plane_path / "stat.npy", allow_pickle=True)
#
#     # Background image
#     mean_imgs[pid] = ops["meanImg"]
#
#     # Full Suite2P ROI list for that plane
#     stat_per_plane[pid] = stat
#
# print("Loaded planes:", list(mean_imgs.keys()))


In [ ]:




### I will plot for onset,

# map of index from dFoF_merged
map_df = dFoF_merged_map
print(map_df.head())

#{'FL1': 1, 'FL2': 2, 'FL3': 3, 'FLB': 4, 'FR1': 5, 'FR2': 6, 'FR3': 7, 'FRB': 8, 'LLB': 9, 'RLB': 10}
# take out NaNs and limit max value to 4 seconds from the neurons that respond to stim X ex 9 = LLB
onset_9 = onset_sec_by_id[10].copy()
mask = (~np.isnan(onset_9))# & (v <= 4)   # True si NO es NaN y v <= 6
onset_9_clean = onset_9[mask]
onset_9_clean[onset_9_clean> 4] = 4


# I am taking the same mask to filter kept indices
kept_indices=results["kept_neuron_indices"]
kept_indices_clean = kept_indices[mask]
# e.g.: BPI = np.load(paths['dfof_dir'] / "L433_f02_BPI_kept.npy")
onset = onset_9_clean # <--- put your actual variable here
print("onset shape:", onset.shape)
assert onset.shape[0] == kept_indices_clean.shape[0], "onset and kept_indices must have same length"
# DataFrame for kept neurons: global_col ↔ onset or other continuous value
sel_df = pd.DataFrame({
    "global_col": kept_indices_clean,
    "onset": onset
})

# Join with map to add plane and filtered_roi_index
merged = sel_df.merge(
    map_df[["plane", "filtered_roi_index", "global_col"]],
    on="global_col",
    how="left"
)
print(sel_df.head())
print("Merged kept+map example:")
print(merged.head())
assert merged["filtered_roi_index"].isna().sum() == 0, "Some kept neurons not in map_df!"

# Initialize dict: plane -> list of (k, BPI)
entries_per_plane = {pid: [] for pid in plane_ids}

for _, row in merged.iterrows():
    plane_label = row["plane"]                 # 'plane0', 'plane1', ...
    k = int(row["filtered_roi_index"])         # filtered ROI index
    value = float(row["onset"])              # value
# store (Suite2P ROI index, BPI) in the list for that plane
    entries_per_plane[plane_label].append((k, value))

for pid in plane_ids:
    print(pid, "n entries:", len(entries_per_plane[pid]))


In [ ]:
def plot_bpi_rois_by_plane(
    plane_ids,
    mean_imgs,
    stat_per_plane,
    entries_per_plane,
    value_norm=None,
    cmap=None,
    value_label="Bout Preference index",
    n_rows=2,
    n_cols=3,
    gamma=0.8,
):
    if cmap is None:
        cmap = plt.cm.winter

    # collect all BPI values to define color scale
    all_vals = []
    for pid in plane_ids:
        for _, v in entries_per_plane.get(pid, []):
            if np.isfinite(v):
                all_vals.append(v)

    if not all_vals:
        raise ValueError("No values to plot.")

    all_vals = np.asarray(all_vals)
    if value_norm is None:
        vmin, vmax = np.nanmin(all_vals), np.nanmax(all_vals)
        if vmin == vmax:
            vmin -= 0.5
            vmax += 0.5
        value_norm = mpl.colors.Normalize(vmin= vmin, vmax= vmax)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 7))
    axes = np.atleast_1d(axes).ravel()

    for i, pid in enumerate(plane_ids):
        if i >= len(axes):
            break

        ax = axes[i]
        mean_img = mean_imgs[pid]
        entries = entries_per_plane.get(pid, [])
        stat_list = stat_per_plane[pid]

        # normalize background (1–99th percentile)
        p1, p99 = np.percentile(mean_img, (1, 99))
        denom = (p99 - p1) if (p99 - p1) != 0 else 1.0
        img_clip = np.clip(mean_img, p1, p99)
        img_norm = (img_clip - p1) / denom
        img_gamma = img_norm ** gamma

        ax.set_facecolor("black")
        ax.imshow(img_gamma, cmap="gray", origin="lower", alpha=0.4)

        # plot each ROI
        for k, val in entries:
            st = stat_list[k]              # Suite2P ROI dict
            xpix = st["xpix"]
            ypix = st["ypix"]
            color = cmap(value_norm(val))
            ax.plot(xpix, ypix, ".", markersize=0.8, color=color, alpha=1)

        ax.set_title(f"{pid} — n={len(entries)}", fontsize=10)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.set_xticks([])
        ax.set_yticks([])

    # hide unused panels
    for j in range(len(plane_ids), len(axes)):
        axes[j].axis("off")

    # colorbar
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=value_norm)
    sm.set_array([])
    last_ax = axes[min(len(plane_ids) - 1, len(axes) - 1)]
    bbox = last_ax.get_position()
    cax = fig.add_axes([bbox.x1 + 0.05, bbox.y0 - 0.015, 0.015, bbox.height])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label(value_label)
    cbar.ax.invert_yaxis()

    plt.tight_layout()
    plt.show()


In [ ]:
plot_bpi_rois_by_plane(
    plane_ids=plane_ids,
    mean_imgs=mean_imgs,
    stat_per_plane=stat_per_plane,
    entries_per_plane=entries_per_plane,
    value_label="onset in seconds",
)


Plot neurons per plane that respond to flicker position left and right and their union

In [ ]:
### plotting neurons rsponsive for Flicker left and right and their union
### create a DataFrame with the union of both sets and label them accordingly, then this will me merget with map_df to get plane and Suite2P ROI index (k)

#{'FL1': 1, 'FL2': 2, 'FL3': 3, 'FLB': 4, 'FR1': 5, 'FR2': 6, 'FR3': 7, 'FRB': 8, 'LLB': 9, 'RLB': 10}


kept_indices=results["kept_neuron_indices"]
flicker_Left= kept_indices[indx_12]
flicker_right = kept_indices[indx_67]

# 1) Union of global indices
global_union = np.unique(np.concatenate([flicker_Left, flicker_right]))
print("Total in union:", global_union.shape[0])

# 2) DataFrame with union
sel_df = pd.DataFrame({"global_col": global_union})

# 3) Merge with map_df to get plane and Suite2P ROI index (k)
merged = sel_df.merge(
    map_df[["plane", "filtered_roi_index", "global_col"]],
    on="global_col",
    how="left",
)

print("Merged A∪B example:")
print(merged.head())
assert merged["filtered_roi_index"].isna().sum() == 0, "Some union neurons not in map_df!"



In [ ]:
# here we label each neuron in the union as "A_only", "B_only", or "both" based on its presence in global_a and global_b which if are flickeres or not

# Turn A and B into sets (for fast membership)
set_a = set(flicker_Left.tolist())
set_b = set(flicker_right.tolist())

def label_category(g):
    in_a = g in set_a
    in_b = g in set_b
    if in_a and in_b:
        return "both"
    elif in_a:
        return "A_only"
    elif in_b:
        return "B_only"
    else:
        return "none"  # should not happen if we used union

merged["category"] = merged["global_col"].apply(label_category)

print(merged["category"].value_counts())
print(merged.head())

In [ ]:
# Now, we can collect Suite2P ROI indices (k) per plane and category

indices_A_only_per_plane = {pid: [] for pid in plane_ids}
indices_B_only_per_plane = {pid: [] for pid in plane_ids}
indices_both_per_plane   = {pid: [] for pid in plane_ids}

for _, row in merged.iterrows():
    plane_label = row["plane"]               # 'plane0', 'plane1', ...
    k = int(row["filtered_roi_index"])       # Suite2P ROI index
    cat = row["category"]

    if cat == "A_only":
        indices_A_only_per_plane[plane_label].append(k)
    elif cat == "B_only":
        indices_B_only_per_plane[plane_label].append(k)
    elif cat == "both":
        indices_both_per_plane[plane_label].append(k)

for pid in plane_ids:
    print(
        pid,
        "A_only:", len(indices_A_only_per_plane[pid]),
        "B_only:", len(indices_B_only_per_plane[pid]),
        "both:",   len(indices_both_per_plane[pid]),
    )


In [ ]:
# Now we can plot the three sets per plane using different colors, we plot the neurons responsive in the anatomical map, per plane, we can plot 3 sets of neurons:

def plot_three_sets_rois_by_plane(
    plane_ids, # name of planes to plot
    mean_imgs, # mean images per plane
    stat_per_plane, # Suite2P stat list per plane
    indices_A_only=None,
    indices_B_only=None,
    indices_both=None,  # could another set
    color_A="cyan",
    color_B="magenta",
    color_both="yellow",
    label_A="Set A",
    label_B="Set B",
    label_both="Shared",
    n_rows=2,
    n_cols=3,
    gamma=0.8,
):
    """
    Plot up to three neuron subsets (A_only, B_only, both) on mean image per plane.

    You can use it with:
        - only indices_A_only                      → single-color plot
        - indices_A_only + indices_B_only          → two-color plot
        - indices_A_only + indices_B_only + both   → three-color plot

    indices_*_per_plane must be dicts:
        indices_X[plane_id] -> list/array of Suite2P ROI indices (k)
    """

    # Ensure dicts (even if None was passed)
    indices_A_only = indices_A_only or {}
    indices_B_only = indices_B_only or {}
    indices_both   = indices_both   or {}

    # Global counts to know which sets actually exist
    nA_total    = sum(len(v) for v in indices_A_only.values())
    nB_total    = sum(len(v) for v in indices_B_only.values())
    nBoth_total = sum(len(v) for v in indices_both.values())

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 7))
    axes = np.atleast_1d(axes).ravel()

    # --- plot each plane ---
    for i, pid in enumerate(plane_ids):
        if i >= len(axes):
            break

        ax = axes[i]
        mean_img = mean_imgs[pid]
        stat_list = stat_per_plane[pid]

        A_idxs    = indices_A_only.get(pid, [])
        B_idxs    = indices_B_only.get(pid, [])
        Both_idxs = indices_both.get(pid, [])

        # normalize background (1–99 percentile)
        p1, p99 = np.percentile(mean_img, (1, 99))
        denom = (p99 - p1) if (p99 - p1) != 0 else 1.0
        img_clip  = np.clip(mean_img, p1, p99)
        img_norm  = (img_clip - p1) / denom
        img_gamma = img_norm ** gamma

        ax.set_facecolor("black")
        ax.imshow(img_gamma, cmap="gray", origin="lower", alpha=0.4)

        # A-only
        for k in A_idxs:
            st = stat_list[int(k)]
            xpix = st["xpix"]
            ypix = st["ypix"]
            ax.plot(xpix, ypix, ".", markersize=0.8, color=color_A, alpha=1)

        # B-only
        for k in B_idxs:
            st = stat_list[int(k)]
            xpix = st["xpix"]
            ypix = st["ypix"]
            ax.plot(xpix, ypix, ".", markersize=0.8, color=color_B, alpha=1)

        # both
        for k in Both_idxs:
            st = stat_list[int(k)]
            xpix = st["xpix"]
            ypix = st["ypix"]
            ax.plot(xpix, ypix, ".", markersize=0.8, color=color_both, alpha=1)

        nA    = len(A_idxs)
        nB    = len(B_idxs)
        nBoth = len(Both_idxs)
        ax.set_title(f"{pid} — {label_A}:{nA}  {label_B}:{nB}  {label_both}:{nBoth}",
                     fontsize=9)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.set_xticks([])
        ax.set_yticks([])

    # --- hide unused axes AND remember first empty one for legend ---
    legend_ax = None
    if len(plane_ids) < len(axes):
        legend_ax = axes[len(plane_ids)]   # first unused slot
    for j in range(len(plane_ids), len(axes)):
        axes[j].axis("off")

    # --- build legend handles ---
    legend_handles = []
    legend_labels  = []

    if nA_total > 0:
        legend_handles.append(Line2D([0], [0], marker='.', linestyle='',
                                     markersize=6, color=color_A))
        legend_labels.append(label_A)

    if nB_total > 0:
        legend_handles.append(Line2D([0], [0], marker='.', linestyle='',
                                     markersize=6, color=color_B))
        legend_labels.append(label_B)

    if nBoth_total > 0:
        legend_handles.append(Line2D([0], [0], marker='.', linestyle='',
                                     markersize=6, color=color_both))
        legend_labels.append(label_both)

    # --- place legend in empty subplot if available, otherwise on first axis ---
    if legend_handles:
        if legend_ax is None:
            legend_ax = axes[0]
        legend_ax.axis("off")  # keep it clean
        legend_ax.legend(legend_handles, legend_labels,
                         loc="center", fontsize=9, frameon=False)

    plt.tight_layout()
    plt.show()


In [ ]:
plot_three_sets_rois_by_plane(
    plane_ids=plane_ids,
    mean_imgs=mean_imgs,
    stat_per_plane=stat_per_plane,
    indices_A_only=indices_A_only_per_plane,
    indices_B_only=indices_B_only_per_plane,
    indices_both=indices_both_per_plane,
    color_A="cyan",
    color_B="magenta",
    color_both="yellow",
    label_A="Bouts",
    label_B="L-R-Rocking",
    label_both="Shared",

)


In [ ]:
### So tomorrow, I clean the code, (load whatever you can in data_loading and  put functons in src)
# I run for several fishes and look for saving, this implyed firs find the raste per fish and found the kepts neuros,
### Then I began buiding the presentation for undertand where I have to begin to generate average interfish stuids
